In [1]:
%pip install -qU \
    langchain \
    langchain-core \
    langchain-community \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    pandas==2.2.3 \
    tabulate \
    pillow \
    transformers \
    accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 116.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/

In [2]:
%pip install -q --no-deps "requests==2.32.4"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.2 MB/s eta 0:00:00


In [1]:
import requests
import torch
import langchain
import langchain_core

print("Requests:", requests.__version__)
print("GPU:", torch.cuda.is_available())
print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Requests: 2.32.4
GPU: True
LangChain: 1.4.3
LangChain Core: 1.6.5
GPU: Tesla T4


In [2]:
import os
import io
import time
import base64
from pathlib import Path

import torch
import fitz
import pandas as pd

from PIL import Image
from IPython.display import display

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("✅ Imports successful")

✅ Imports successful


/tmp/ipykernel_1364/2391528805.py:19: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Embedding model loaded")
print("Device:", device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded
Device: cuda


In [5]:
PDF_PATH = Path("NovaCore_Multimodal_Company_Report_2026.pdf")

# Fallback for Colab/local mounted file
if not PDF_PATH.exists():
    fallback = Path("/mnt/data/NovaCore_Multimodal_Company_Report_2026.pdf")
    if fallback.exists():
        PDF_PATH = fallback

assert PDF_PATH.exists(), (
    "PDF not found. Upload NovaCore_Multimodal_Company_Report_2026.pdf "
    "to Colab before continuing."
)

# Models
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Directory for extracted images
IMAGE_DIR = Path("novacore_extracted_images")
IMAGE_DIR.mkdir(exist_ok=True)

print("PDF:", PDF_PATH)
print("Embedding model:", EMBEDDING_MODEL)
print("Image directory:", IMAGE_DIR)

PDF: NovaCore_Multimodal_Company_Report_2026.pdf
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Image directory: novacore_extracted_images


In [6]:
def image_to_data_uri(image_path):
    image_path = Path(image_path)

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((1600, 1600))

        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    return f"data:image/jpeg;base64,{encoded}"

In [7]:
%pip install -qU transformers accelerate qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 38.5 MB/s eta 0:00:00


In [8]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

VISION_MODEL = "Qwen/Qwen2-VL-2B-Instruct"

vision_processor = AutoProcessor.from_pretrained(VISION_MODEL)

vision_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VISION_MODEL,
    torch_dtype="auto",
    device_map="auto",
)

print("✅ Vision model loaded")
print("Model:", VISION_MODEL)

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

✅ Vision model loaded
Model: Qwen/Qwen2-VL-2B-Instruct


In [9]:
def summarize_visual(image_path, page_number):
    image_path = Path(image_path)

    prompt = f"""
This visual was extracted from page {page_number} of the NovaCore FY2026 company report.

Describe the useful business information visible in the visual.

If it is a chart or graph:
- mention important values
- mention highest/lowest values
- mention the main trend

If it is a diagram:
- identify important components
- explain the flow or relationships

If it is a normal business image:
- describe the useful factual information

Keep the description concise and factual.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": str(image_path),
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]

    text = vision_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = vision_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(vision_model.device)

    generated_ids = vision_model.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=False,
    )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = vision_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return output_text.strip()


print("✅ Local visual summarizer ready")

✅ Local visual summarizer ready


In [10]:
print("PDF exists:", PDF_PATH.exists())
print("PDF path:", PDF_PATH)
print("Image directory:", IMAGE_DIR)

PDF exists: True
PDF path: NovaCore_Multimodal_Company_Report_2026.pdf
Image directory: novacore_extracted_images


In [11]:
def extract_multimodal_documents(pdf_path):
    pdf = fitz.open(str(pdf_path))
    documents = []
    extracted_xrefs = set()

    for page_index in range(len(pdf)):
        page = pdf[page_index]
        page_number = page_index + 1

        print(f"Processing page {page_number}...")

        # --------------------------------------------------
        # 1. Extract normal text
        # --------------------------------------------------
        text = page.get_text("text").strip()

        if text:
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "page": page_number,
                        "modality": "text",
                        "source": pdf_path.name,
                    },
                )
            )

        # --------------------------------------------------
        # 2. Extract tables
        # --------------------------------------------------
        try:
            tables = page.find_tables().tables

            for table_number, table in enumerate(
                tables,
                start=1,
            ):
                df = table.to_pandas()

                if not df.empty:
                    table_text = df.to_markdown(index=False)

                    documents.append(
                        Document(
                            page_content=table_text,
                            metadata={
                                "page": page_number,
                                "modality": "table",
                                "table_number": table_number,
                                "source": pdf_path.name,
                            },
                        )
                    )

        except Exception as error:
            print(
                "Table extraction warning:",
                error,
            )

        # --------------------------------------------------
        # 3. Extract images
        # --------------------------------------------------
        for image_number, image_info in enumerate(
            page.get_images(full=True),
            start=1,
        ):
            xref = image_info[0]

            if xref in extracted_xrefs:
                continue

            extracted_xrefs.add(xref)

            image_data = pdf.extract_image(xref)

            image_bytes = image_data["image"]
            image_extension = image_data["ext"]

            image_path = (
                IMAGE_DIR
                / f"page_{page_number}_image_{image_number}.{image_extension}"
            )

            image_path.write_bytes(image_bytes)

            # --------------------------------------------------
            # 4. Summarize image using local Qwen2-VL
            # --------------------------------------------------
            try:
                summary = summarize_visual(
                    image_path=image_path,
                    page_number=page_number,
                )

                documents.append(
                    Document(
                        page_content=summary,
                        metadata={
                            "page": page_number,
                            "modality": "visual",
                            "image_path": str(image_path),
                            "source": pdf_path.name,
                        },
                    )
                )

            except Exception as error:
                print(
                    f"Vision warning on page {page_number}:",
                    error,
                )

    pdf.close()

    return documents


# Run multimodal PDF extraction
documents = extract_multimodal_documents(PDF_PATH)

print()
print("Total LangChain documents:", len(documents))

Processing page 1...
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Processing page 2...
Processing page 3...
Processing page 4...
Processing page 5...
Processing page 6...
Processing page 7...
Processing page 8...
Processing page 9...

Total LangChain documents: 24


In [12]:
# Create local FAISS vector store from all extracted documents

vectorstore = FAISS.from_documents(
    documents,
    embeddings,
)

print("✅ FAISS vector store created")
print("Documents indexed:", len(documents))

✅ FAISS vector store created
Documents indexed: 24


In [13]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("✅ Retriever ready")

✅ Retriever ready


In [14]:
query = "Which region had the highest revenue growth?"

retrieved_docs = retriever.invoke(query)

for i, doc in enumerate(retrieved_docs, start=1):
    print("=" * 80)
    print(
        f"Result {i} | "
        f"Page {doc.metadata.get('page')} | "
        f"Type: {doc.metadata.get('modality')}"
    )
    print(doc.page_content[:700])
    print()

Result 1 | Page 4 | Type: table
| Region        | 2026 revenue   | YoY growth   |   Enterprise customers | CSAT   |
|:--------------|:---------------|:-------------|-----------------------:|:-------|
| Europe        | $41.2M         | 31%          |                    118 | 4.7/5  |
| North America | $38.6M         | 27%          |                    104 | 4.6/5  |
| Asia Pacific  | $34.9M         | 24%          |                     96 | 4.8/5  |
| Middle East   | $17.3M         | 18%          |                     51 | 4.5/5  |

Result 2 | Page 4 | Type: visual
The visual is a bar chart titled "2026 Year-over-Year Revenue Growth by Region." The x-axis represents the regions, with Europe at the top and Middle East at the bottom. The y-axis represents growth in percentage, ranging from 0% to 35%.

The main trend visible in the chart is that Europe has the highest growth rate at 31%, followed by North America at 27%, Asia Pacific at 24%, and Middle East at 18%.

The highest growth rate 

In [15]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline

TEXT_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)

text_model = AutoModelForCausalLM.from_pretrained(
    TEXT_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)

text_pipeline = pipeline(
    "text-generation",
    model=text_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
)

text_llm = HuggingFacePipeline(
    pipeline=text_pipeline
)

print("✅ Local text LLM loaded")
print("Model:", TEXT_MODEL)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Local text LLM loaded
Model: Qwen/Qwen2.5-3B-Instruct


In [16]:
rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant answering questions about the
NovaCore Systems FY2026 company report.

Use ONLY the retrieved context below.

If the answer is not available in the context, say:
"I could not find that information in the report."

Mention page numbers when possible.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""
)

text_rag_chain = (
    rag_prompt
    | text_llm
    | StrOutputParser()
)

print("✅ Text RAG chain ready")

✅ Text RAG chain ready


In [17]:
def format_context(retrieved_docs):
    parts = []

    for doc in retrieved_docs:
        page = doc.metadata.get("page")
        modality = doc.metadata.get("modality")

        parts.append(
            f"[Page {page} | {modality.upper()}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(parts)


print("✅ Context formatter ready")

✅ Context formatter ready


In [18]:
def answer_with_vision(question, context, image_paths):
    prompt = f"""
You are a helpful assistant answering questions about the
NovaCore Systems FY2026 company report.

Use ONLY the provided report context and images.

If the answer is not available in the provided information,
say:
"I could not find that information in the report."

Mention page numbers when possible.

REPORT CONTEXT:
{context}

QUESTION:
{question}

Answer clearly and concisely.
"""

    content = [
        {
            "type": "text",
            "text": prompt,
        }
    ]

    # Add retrieved images
    for image_path in image_paths:
        content.append(
            {
                "type": "image",
                "image": str(image_path),
            }
        )

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]

    # Convert messages into the format expected by Qwen2-VL
    text = vision_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = vision_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(vision_model.device)

    generated_ids = vision_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
    )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs.input_ids,
            generated_ids,
        )
    ]

    output_text = vision_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return output_text.strip()


print("✅ Multimodal vision answerer ready")

✅ Multimodal vision answerer ready


In [19]:
def ask_rag(question):
    # --------------------------------------------------
    # 1. Retrieve relevant documents from FAISS
    # --------------------------------------------------
    retrieved_docs = retriever.invoke(question)

    # --------------------------------------------------
    # 2. Format retrieved context
    # --------------------------------------------------
    context = format_context(retrieved_docs)

    # --------------------------------------------------
    # 3. Find original images among retrieved documents
    # --------------------------------------------------
    image_paths = []

    for doc in retrieved_docs:
        image_path = doc.metadata.get("image_path")

        if image_path:
            image_path = Path(image_path)

            if image_path.exists():
                image_paths.append(image_path)

    # Remove duplicate image paths
    image_paths = list(dict.fromkeys(image_paths))

    # --------------------------------------------------
    # 4. Generate answer
    # --------------------------------------------------
    if image_paths:
        answer = answer_with_vision(
            question=question,
            context=context,
            image_paths=image_paths,
        )
    else:
        answer = text_rag_chain.invoke(
            {
                "context": context,
                "question": question,
            }
        )

    # --------------------------------------------------
    # 5. Display answer
    # --------------------------------------------------
    print("=" * 80)
    print("QUESTION")
    print(question)

    print("\n" + "=" * 80)
    print("ANSWER")
    print(answer)

    # --------------------------------------------------
    # 6. Display retrieved sources
    # --------------------------------------------------
    print("\n" + "=" * 80)
    print("RETRIEVED SOURCES")

    for i, doc in enumerate(retrieved_docs, start=1):
        page = doc.metadata.get("page")
        modality = doc.metadata.get("modality")

        print(
            f"{i}. Page {page} | "
            f"Type: {modality}"
        )

    print("=" * 80)

    return answer

In [20]:
print("✅ Final RAG function ready")

✅ Final RAG function ready


In [21]:
question = "Which region had the highest revenue growth?"

answer = ask_rag(question)

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.09 GiB. GPU 0 has a total capacity of 14.56 GiB of which 945.81 MiB is free. Including non-PyTorch memory, this process has 13.64 GiB memory in use. Of the allocated memory 12.30 GiB is allocated by PyTorch, and 1.21 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [22]:
import gc
import torch

del text_pipeline
del text_llm
del text_model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print("✅ Old text model unloaded")
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

✅ Old text model unloaded
GPU memory allocated: 10.1 GB


In [23]:
%pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.2 MB/s eta 0:00:00


In [24]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from langchain_huggingface import HuggingFacePipeline

TEXT_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

text_model = AutoModelForCausalLM.from_pretrained(
    TEXT_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)

text_pipeline = pipeline(
    "text-generation",
    model=text_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    return_full_text=False,
)

text_llm = HuggingFacePipeline(
    pipeline=text_pipeline
)

print("✅ 4-bit text LLM loaded")
print("Model:", TEXT_MODEL)
print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ 4-bit text LLM loaded
Model: Qwen/Qwen2.5-3B-Instruct
GPU memory allocated: 12.15 GB


In [25]:
import torch

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
free = total - allocated

print(f"GPU total:     {total:.2f} GB")
print(f"GPU allocated: {allocated:.2f} GB")
print(f"GPU reserved:  {reserved:.2f} GB")
print(f"GPU available: {free:.2f} GB")

GPU total:     14.56 GB
GPU allocated: 12.15 GB
GPU reserved:  12.26 GB
GPU available: 2.42 GB


In [26]:
import gc
import torch

del text_pipeline
del text_llm
del text_model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

GPU memory allocated: 10.1 GB


In [27]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline,
)

from langchain_huggingface import HuggingFacePipeline

TEXT_MODEL = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True,
)

text_model = AutoModelForCausalLM.from_pretrained(
    TEXT_MODEL,
    quantization_config=quant_config,
    device_map={
        "": "cpu",
    },
)

text_pipeline = pipeline(
    "text-generation",
    model=text_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=False,
    return_full_text=False,
)

text_llm = HuggingFacePipeline(
    pipeline=text_pipeline
)

print("✅ CPU-offloaded 4-bit text LLM loaded")

[transformers] You are loading your model using eetq but no linear modules were found in your model. Please double check your model architecture, or submit an issue on github if you think this is a bug.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 44.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 13.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.29 GiB is allocated by PyTorch, and 132.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [28]:
import gc
import torch

# Remove failed text-model objects if they exist
for name in [
    "text_pipeline",
    "text_llm",
    "text_model",
    "tokenizer",
]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print(
    "GPU reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

GPU allocated: 14.14 GB
GPU reserved: 14.42 GB


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB"
)

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [2]:
import os
import io
import time
import base64
from pathlib import Path

import torch
import fitz
import pandas as pd

from PIL import Image
from IPython.display import display

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("✅ Imports restored")

✅ Imports restored


/tmp/ipykernel_12881/1634756703.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

print("✅ Embedding model loaded")
print("Device:", device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Embedding model loaded
Device: cuda


In [4]:
PDF_PATH = Path("/content/NovaCore_Multimodal_Company_Report_2026.pdf")

IMAGE_DIR = Path("/content/novacore_extracted_images")
IMAGE_DIR.mkdir(exist_ok=True)

assert PDF_PATH.exists(), (
    f"PDF not found: {PDF_PATH}"
)

print("✅ Paths restored")
print("PDF:", PDF_PATH)
print("Image directory:", IMAGE_DIR)

✅ Paths restored
PDF: /content/NovaCore_Multimodal_Company_Report_2026.pdf
Image directory: /content/novacore_extracted_images


In [5]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

VISION_MODEL = "Qwen/Qwen2-VL-2B-Instruct"

vision_processor = AutoProcessor.from_pretrained(
    VISION_MODEL
)

vision_model = Qwen2VLForConditionalGeneration.from_pretrained(
    VISION_MODEL,
    torch_dtype="auto",
    device_map="auto",
)

vision_model.eval()

print("✅ Vision model restored")
print("Model:", VISION_MODEL)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

✅ Vision model restored
Model: Qwen/Qwen2-VL-2B-Instruct
GPU memory allocated: 4.2 GB


In [6]:
def image_to_data_uri(image_path):
    image_path = Path(image_path)

    with Image.open(image_path) as img:
        img = img.convert("RGB")
        img.thumbnail((1600, 1600))

        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=85)

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    return f"data:image/jpeg;base64,{encoded}"

In [7]:
def summarize_visual(image_path, page_number):
    image_path = Path(image_path)

    prompt = f"""
This visual was extracted from page {page_number} of the NovaCore FY2026 company report.

Describe the useful business information visible in the visual.

If it is a chart or graph:
- mention important values
- mention highest/lowest values
- mention the main trend

If it is a diagram:
- identify important components
- explain the flow or relationships

If it is a normal business image:
- describe the useful factual information

Keep the description concise and factual.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": str(image_path),
                },
                {
                    "type": "text",
                    "text": prompt,
                },
            ],
        }
    ]

    text = vision_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = vision_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(vision_model.device)

    with torch.inference_mode():
        generated_ids = vision_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs.input_ids,
            generated_ids,
        )
    ]

    output_text = vision_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return output_text.strip()


print("✅ Visual summarizer restored")

✅ Visual summarizer restored


In [8]:
def extract_multimodal_documents(pdf_path):
    pdf = fitz.open(str(pdf_path))
    documents = []
    extracted_xrefs = set()

    for page_index in range(len(pdf)):
        page = pdf[page_index]
        page_number = page_index + 1

        print(f"Processing page {page_number}...")

        # --------------------------------------------------
        # 1. Extract normal text
        # --------------------------------------------------
        text = page.get_text("text").strip()

        if text:
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "page": page_number,
                        "modality": "text",
                        "source": pdf_path.name,
                    },
                )
            )

        # --------------------------------------------------
        # 2. Extract tables
        # --------------------------------------------------
        try:
            tables = page.find_tables().tables

            for table_number, table in enumerate(
                tables,
                start=1,
            ):
                df = table.to_pandas()

                if not df.empty:
                    table_text = df.to_markdown(index=False)

                    documents.append(
                        Document(
                            page_content=table_text,
                            metadata={
                                "page": page_number,
                                "modality": "table",
                                "table_number": table_number,
                                "source": pdf_path.name,
                            },
                        )
                    )

        except Exception as error:
            print(
                "Table extraction warning:",
                error,
            )

        # --------------------------------------------------
        # 3. Extract images
        # --------------------------------------------------
        for image_number, image_info in enumerate(
            page.get_images(full=True),
            start=1,
        ):
            xref = image_info[0]

            if xref in extracted_xrefs:
                continue

            extracted_xrefs.add(xref)

            image_data = pdf.extract_image(xref)

            image_bytes = image_data["image"]
            image_extension = image_data["ext"]

            image_path = (
                IMAGE_DIR
                / f"page_{page_number}_image_{image_number}.{image_extension}"
            )

            image_path.write_bytes(image_bytes)

            # --------------------------------------------------
            # 4. Summarize image with local Qwen2-VL
            # --------------------------------------------------
            try:
                summary = summarize_visual(
                    image_path=image_path,
                    page_number=page_number,
                )

                documents.append(
                    Document(
                        page_content=summary,
                        metadata={
                            "page": page_number,
                            "modality": "visual",
                            "image_path": str(image_path),
                            "source": pdf_path.name,
                        },
                    )
                )

            except Exception as error:
                print(
                    f"Vision warning on page {page_number}:",
                    error,
                )

    pdf.close()

    return documents


documents = extract_multimodal_documents(PDF_PATH)

print()
print("Total LangChain documents:", len(documents))

Processing page 1...
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Processing page 2...
Processing page 3...
Processing page 4...
Processing page 5...
Processing page 6...
Processing page 7...
Processing page 8...
Processing page 9...

Total LangChain documents: 24


In [9]:
vectorstore = FAISS.from_documents(
    documents,
    embeddings,
)

print("✅ FAISS vector store recreated")
print("Documents indexed:", len(documents))

✅ FAISS vector store recreated
Documents indexed: 24


In [10]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("✅ Retriever recreated")

✅ Retriever recreated


In [11]:
def answer_with_vision(question, context, image_paths):
    prompt = f"""
You are a helpful assistant answering questions about the
NovaCore Systems FY2026 company report.

Use ONLY the provided report context and images.

If the answer is not available in the provided information,
say:
"I could not find that information in the report."

Mention page numbers when possible.

REPORT CONTEXT:
{context}

QUESTION:
{question}

Answer clearly and concisely.
"""

    content = [
        {
            "type": "text",
            "text": prompt,
        }
    ]

    # Add retrieved original images
    for image_path in image_paths:
        content.append(
            {
                "type": "image",
                "image": str(image_path),
            }
        )

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]

    text = vision_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = vision_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(vision_model.device)

    with torch.inference_mode():
        generated_ids = vision_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs.input_ids,
            generated_ids,
        )
    ]

    answer = vision_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return answer.strip()


print("✅ Final multimodal answer function ready")

✅ Final multimodal answer function ready


In [12]:
def ask_rag(question):
    # --------------------------------------------------
    # 1. Retrieve relevant documents from FAISS
    # --------------------------------------------------
    retrieved_docs = retriever.invoke(question)

    # --------------------------------------------------
    # 2. Format retrieved context
    # --------------------------------------------------
    parts = []

    for doc in retrieved_docs:
        page = doc.metadata.get("page")
        modality = doc.metadata.get("modality")

        parts.append(
            f"[Page {page} | {modality.upper()}]\n"
            f"{doc.page_content}"
        )

    context = "\n\n".join(parts)

    # --------------------------------------------------
    # 3. Find original images
    # --------------------------------------------------
    image_paths = []

    for doc in retrieved_docs:
        image_path = doc.metadata.get("image_path")

        if image_path:
            image_path = Path(image_path)

            if image_path.exists():
                image_paths.append(image_path)

    image_paths = list(dict.fromkeys(image_paths))

    # --------------------------------------------------
    # 4. Generate final answer using Qwen2-VL
    # --------------------------------------------------
    answer = answer_with_vision(
        question=question,
        context=context,
        image_paths=image_paths,
    )

    # --------------------------------------------------
    # 5. Display answer
    # --------------------------------------------------
    print("=" * 80)
    print("QUESTION")
    print(question)

    print("\n" + "=" * 80)
    print("ANSWER")
    print(answer)

    # --------------------------------------------------
    # 6. Display retrieved sources
    # --------------------------------------------------
    print("\n" + "=" * 80)
    print("RETRIEVED SOURCES")

    for i, doc in enumerate(retrieved_docs, start=1):
        print(
            f"{i}. Page {doc.metadata.get('page')} | "
            f"Type: {doc.metadata.get('modality')}"
        )

    print("=" * 80)

    return answer


print("✅ Complete Multimodal RAG pipeline ready")

✅ Complete Multimodal RAG pipeline ready


In [13]:
question = "Which region had the highest revenue growth?"

answer = ask_rag(question)

QUESTION
Which region had the highest revenue growth?

ANSWER
Europe

RETRIEVED SOURCES
1. Page 4 | Type: table
2. Page 4 | Type: visual
3. Page 2 | Type: table
4. Page 4 | Type: text
5. Page 3 | Type: table


In [14]:
def answer_with_vision(question, context, image_paths):
    prompt = f"""
You are answering questions about the NovaCore Systems FY2026 company report.

Use ONLY the provided report context and images.

Rules:
- Give a complete, direct answer.
- Include important numbers when they are available.
- Mention the relevant page number when possible.
- Do not invent information.
- If the information is not available, say:
  "I could not find that information in the report."

REPORT CONTEXT:
{context}

QUESTION:
{question}

Give a concise answer in 1-3 sentences.
"""

    content = [
        {
            "type": "text",
            "text": prompt,
        }
    ]

    for image_path in image_paths:
        content.append(
            {
                "type": "image",
                "image": str(image_path),
            }
        )

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]

    text = vision_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = vision_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(vision_model.device)

    with torch.inference_mode():
        generated_ids = vision_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs.input_ids,
            generated_ids,
        )
    ]

    answer = vision_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return answer.strip()


print("✅ Improved answer generator ready")

✅ Improved answer generator ready


In [15]:
question = "Which region had the highest revenue growth?"

answer = ask_rag(question)

QUESTION
Which region had the highest revenue growth?

ANSWER
Europe

RETRIEVED SOURCES
1. Page 4 | Type: table
2. Page 4 | Type: visual
3. Page 2 | Type: table
4. Page 4 | Type: text
5. Page 3 | Type: table


In [16]:
def answer_with_vision(question, context, image_paths):
    prompt = f"""
Answer the question using the report evidence below.

IMPORTANT:
1. Give the answer directly.
2. Include the relevant number or measurement if one is present.
3. Explain why the answer is correct in one short sentence.
4. Mention the page number from the context.
5. Do not answer with only a single word or name.
6. Do not use information outside the supplied report evidence.

REPORT EVIDENCE:
{context}

QUESTION:
{question}

ANSWER:
"""

    content = [
        {
            "type": "text",
            "text": prompt,
        }
    ]

    for image_path in image_paths:
        content.append(
            {
                "type": "image",
                "image": str(image_path),
            }
        )

    messages = [
        {
            "role": "user",
            "content": content,
        }
    ]

    text = vision_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = vision_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

    inputs = inputs.to(vision_model.device)

    with torch.inference_mode():
        generated_ids = vision_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(
            inputs.input_ids,
            generated_ids,
        )
    ]

    answer = vision_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    return answer.strip()


print("✅ Answer generator updated")

✅ Answer generator updated


In [17]:
ask_rag("Which region had the highest revenue growth?")

QUESTION
Which region had the highest revenue growth?

ANSWER
Europe

RETRIEVED SOURCES
1. Page 4 | Type: table
2. Page 4 | Type: visual
3. Page 2 | Type: table
4. Page 4 | Type: text
5. Page 3 | Type: table


'Europe'

In [18]:
ask_rag("What was NovaCore's FY2026 revenue?")

QUESTION
What was NovaCore's FY2026 revenue?

ANSWER
$132.0M

RETRIEVED SOURCES
1. Page 3 | Type: text
2. Page 2 | Type: text
3. Page 5 | Type: text
4. Page 9 | Type: text
5. Page 5 | Type: visual


'$132.0M'

In [19]:
ask_rag("What was the gross margin of the AtlasEdge X1?")

QUESTION
What was the gross margin of the AtlasEdge X1?

ANSWER
46%

RETRIEVED SOURCES
1. Page 3 | Type: text
2. Page 5 | Type: visual
3. Page 2 | Type: table
4. Page 5 | Type: text
5. Page 5 | Type: table


'46%'

In [20]:
ask_rag("What are the main components and flow shown in the supply-chain diagram?")

QUESTION
What are the main components and flow shown in the supply-chain diagram?

ANSWER
The main components and flow shown in the supply-chain diagram are:

1. Component Suppliers located in Malaysia and Taiwan.
2. Assembly in Penang, Malaysia.
3. Quality Lab in Singapore.
4. Regional Hubs in Rotterdam and Dubai.
5. Enterprise Customers in 42 countries.

The critical control point highlighted is the Quality Lab, where every production batch must pass thermal, network, and firmware tests before export.

RETRIEVED SOURCES
1. Page 6 | Type: visual
2. Page 6 | Type: text
3. Page 8 | Type: table
4. Page 9 | Type: table
5. Page 5 | Type: table


'The main components and flow shown in the supply-chain diagram are:\n\n1. Component Suppliers located in Malaysia and Taiwan.\n2. Assembly in Penang, Malaysia.\n3. Quality Lab in Singapore.\n4. Regional Hubs in Rotterdam and Dubai.\n5. Enterprise Customers in 42 countries.\n\nThe critical control point highlighted is the Quality Lab, where every production batch must pass thermal, network, and firmware tests before export.'

In [21]:
# Save the FAISS vector store locally

FAISS_PATH = "/content/novacore_faiss_index"

vectorstore.save_local(FAISS_PATH)

print("✅ FAISS index saved")
print("Location:", FAISS_PATH)

✅ FAISS index saved
Location: /content/novacore_faiss_index


In [22]:
from pathlib import Path

faiss_path = Path(FAISS_PATH)

print("Files saved:")
for file in faiss_path.iterdir():
    print("-", file.name)

Files saved:
- index.faiss
- index.pkl


In [23]:
# Reload the saved FAISS index

vectorstore = FAISS.load_local(
    FAISS_PATH,
    embeddings,
    allow_dangerous_deserialization=True,
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print("✅ FAISS index reloaded successfully")

✅ FAISS index reloaded successfully


In [24]:
docs = retriever.invoke(
    "Which region had the highest revenue growth?"
)

print("Retrieved documents:", len(docs))

for i, doc in enumerate(docs, start=1):
    print(
        f"{i}. Page {doc.metadata.get('page')} | "
        f"{doc.metadata.get('modality')}"
    )

Retrieved documents: 5
1. Page 4 | table
2. Page 4 | visual
3. Page 2 | table
4. Page 4 | text
5. Page 3 | table


In [25]:
demo_questions = [
    "Which region had the highest revenue growth?",
    "What are the main components and flow shown in the supply-chain diagram?",
    "What was NovaCore's revenue in Europe?",
    "Which region had the highest customer satisfaction?",
    "What are the main production and quality-control locations?",
    "What is the company's electricity mix?",
]

for i, question in enumerate(demo_questions, start=1):
    print(f"{i}. {question}")

1. Which region had the highest revenue growth?
2. What are the main components and flow shown in the supply-chain diagram?
3. What was NovaCore's revenue in Europe?
4. Which region had the highest customer satisfaction?
5. What are the main production and quality-control locations?
6. What is the company's electricity mix?


In [26]:
answer = ask_rag(demo_questions[0])

QUESTION
Which region had the highest revenue growth?

ANSWER
Europe

RETRIEVED SOURCES
1. Page 4 | Type: table
2. Page 4 | Type: visual
3. Page 2 | Type: table
4. Page 4 | Type: text
5. Page 3 | Type: table


In [27]:
from IPython.display import clear_output

clear_output(wait=True)

In [28]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("✅ GPU memory cleaned")

✅ GPU memory cleaned


In [29]:
!nvidia-smi

Tue Sep 29 13:48:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             34W /   70W |    4509MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [30]:
requirements = """\
torch
transformers
accelerate
qwen-vl-utils
pymupdf
pandas
pillow
sentence-transformers
langchain
langchain-core
langchain-huggingface
langchain-community
faiss-cpu
"""

with open("/content/requirements.txt", "w") as f:
    f.write(requirements)

print("✅ requirements.txt created")

✅ requirements.txt created


In [31]:
!cat /content/requirements.txt

torch
transformers
accelerate
qwen-vl-utils
pymupdf
pandas
pillow
sentence-transformers
langchain
langchain-core
langchain-huggingface
langchain-community
faiss-cpu


In [32]:
readme = """# Multimodal RAG with Qwen2-VL and FAISS

## Overview

This project implements a Multimodal Retrieval-Augmented Generation (RAG) system for answering questions from a company PDF report.

Unlike a traditional text-only RAG system, this project processes:

- Text
- Tables
- Images
- Charts
- Diagrams

Visual content is summarized using Qwen2-VL and stored along with the extracted text and tables. Sentence Transformers are used to create embeddings, while FAISS provides local vector similarity search.

## Architecture

PDF Report
    |
    +--> Text extraction
    |
    +--> Table extraction
    |
    +--> Image extraction
              |
              v
        Qwen2-VL-2B
        Visual summaries
              |
              v
       LangChain Documents
              |
              v
    Sentence Transformers
              |
              v
            FAISS
              |
              v
        Similarity Retrieval
              |
              v
        Retrieved Evidence
              |
              v
        Qwen2-VL-2B
              |
              v
           Answer

## Technologies

- Python
- PyMuPDF
- LangChain
- Qwen2-VL-2B-Instruct
- Hugging Face Transformers
- Sentence Transformers
- FAISS
- Pandas
- Pillow
- Google Colab / NVIDIA T4 GPU

## Multimodal Processing

The PDF is processed page by page.

### Text

Text is extracted directly from each PDF page.

### Tables

Tables are detected and converted into Markdown-formatted text.

### Images and Charts

Images are extracted from the PDF and passed to Qwen2-VL.

The model generates concise factual descriptions of charts, diagrams, and other business visuals.

These descriptions are stored as LangChain documents together with metadata such as:

- Page number
- Modality
- Image path
- Source PDF

## Retrieval

The extracted documents are embedded using:

`sentence-transformers/all-MiniLM-L6-v2`

The embeddings are stored in a FAISS vector index.

For each question, the retriever returns the most relevant documents.

## Generation

The retrieved text, tables, visual summaries, and relevant images are provided to Qwen2-VL-2B.

The model generates an answer using the retrieved report evidence.

## Example Questions

1. Which region had the highest revenue growth?
2. What are the main components and flow shown in the supply-chain diagram?
3. What was NovaCore's revenue in Europe?
4. Which region had the highest customer satisfaction?
5. What are the main production and quality-control locations?
6. What is the company's electricity mix?

## Example

Question:

Which region had the highest revenue growth?

Answer:

Europe

The system retrieved relevant evidence from the report, including the Page 4 table and visual.

## Vector Store Persistence

The FAISS index can be saved locally and reloaded later.

Saved files:

- `index.faiss`
- `index.pkl`

This avoids rebuilding the vector store every time the notebook is restarted.

## Project Structure

```text
multimodal-rag/
|
├── multimodal_rag.ipynb
├── requirements.txt
├── README.md
├── data/
└── novacore_extracted_images/

_IncompleteInputError: incomplete input (3353128591.py, line 1)

In [33]:
from pathlib import Path

readme_lines = [
    "# Multimodal RAG with Qwen2-VL and FAISS",
    "",
    "## Overview",
    "",
    "This project implements a Multimodal Retrieval-Augmented Generation (RAG) system for answering questions from a company PDF report.",
    "",
    "The system processes text, tables, images, charts, and diagrams.",
    "",
    "Visual content is summarized using Qwen2-VL-2B. Sentence Transformers are used to create embeddings, and FAISS is used for local vector similarity search.",
    "",
    "## Architecture",
    "",
    "PDF Report",
    "    |",
    "    +--> Text extraction",
    "    +--> Table extraction",
    "    +--> Image extraction",
    "              |",
    "              v",
    "        Qwen2-VL-2B",
    "        Visual summaries",
    "              |",
    "              v",
    "       LangChain Documents",
    "              |",
    "              v",
    "    Sentence Transformers",
    "              |",
    "              v",
    "            FAISS",
    "              |",
    "              v",
    "        Similarity Retrieval",
    "              |",
    "              v",
    "        Retrieved Evidence",
    "              |",
    "              v",
    "        Qwen2-VL-2B",
    "              |",
    "              v",
    "           Answer",
    "",
    "## Technologies",
    "",
    "- Python",
    "- PyMuPDF",
    "- LangChain",
    "- Qwen2-VL-2B-Instruct",
    "- Hugging Face Transformers",
    "- Sentence Transformers",
    "- FAISS",
    "- Pandas",
    "- Pillow",
    "- Google Colab / NVIDIA T4 GPU",
    "",
    "## Multimodal Processing",
    "",
    "The PDF is processed page by page.",
    "",
    "### Text",
    "",
    "Text is extracted directly from each PDF page.",
    "",
    "### Tables",
    "",
    "Tables are detected and converted into Markdown-formatted text.",
    "",
    "### Images and Charts",
    "",
    "Images are extracted from the PDF and passed to Qwen2-VL.",
    "",
    "The model generates concise factual descriptions of charts, diagrams, and other business visuals.",
    "",
    "These descriptions are stored as LangChain documents together with metadata such as page number, modality, image path, and source PDF.",
    "",
    "## Retrieval",
    "",
    "The extracted documents are embedded using:",
    "",
    "`sentence-transformers/all-MiniLM-L6-v2`",
    "",
    "The embeddings are stored in a FAISS vector index.",
    "",
    "For each question, the retriever returns the most relevant documents.",
    "",
    "## Generation",
    "",
    "The retrieved text, tables, visual summaries, and relevant images are provided to Qwen2-VL-2B.",
    "",
    "The model generates an answer using the retrieved report evidence.",
    "",
    "## Example Questions",
    "",
    "1. Which region had the highest revenue growth?",
    "2. What are the main components and flow shown in the supply-chain diagram?",
    "3. What was NovaCore's revenue in Europe?",
    "4. Which region had the highest customer satisfaction?",
    "5. What are the main production and quality-control locations?",
    "6. What is the company's electricity mix?",
    "",
    "## Example Result",
    "",
    "Question: Which region had the highest revenue growth?",
    "",
    "Answer: Europe",
    "",
    "The system retrieved relevant evidence from the report, including Page 4 table and visual content.",
    "",
    "## Vector Store Persistence",
    "",
    "The FAISS index can be saved locally and reloaded later.",
    "",
    "Saved files:",
    "",
    "- index.faiss",
    "- index.pkl",
    "",
    "This avoids rebuilding the vector store every time the notebook is restarted.",
    "",
    "## Project Structure",
    "",
    "```text",
    "multimodal-rag/",
    "|",
    "+-- multimodal_rag.ipynb",
    "+-- requirements.txt",
    "+-- README.md",
    "+-- data/",
    "+-- novacore_extracted_images/",
    "```",
    "",
    "## Running the Project",
    "",
    "1. Open the notebook in Google Colab.",
    "2. Enable a GPU runtime.",
    "3. Install the dependencies from requirements.txt.",
    "4. Place the PDF report in the expected path.",
    "5. Run the notebook cells in order.",
    "6. The system extracts multimodal content.",
    "7. The FAISS vector store is created.",
    "8. Ask questions using ask_rag().",
    "",
    "## Hardware",
    "",
    "The project was tested using an NVIDIA Tesla T4 GPU in Google Colab.",
    "",
    "Qwen2-VL-2B was selected to keep GPU memory usage practical while supporting multimodal processing.",
    "",
    "## Future Improvements",
    "",
    "- Add document chunking for larger reports.",
    "- Add reranking for improved retrieval.",
    "- Add conversational memory.",
    "- Add a web or Streamlit interface.",
    "- Add citations directly to generated answers.",
    "- Use a stronger multimodal model when more GPU memory is available.",
]

readme = "\n".join(readme_lines)

Path("/content/README.md").write_text(
    readme,
    encoding="utf-8"
)

print("✅ README.md created successfully")

✅ README.md created successfully


In [34]:
print(Path("/content/README.md").read_text(encoding="utf-8")[:2000])

# Multimodal RAG with Qwen2-VL and FAISS

## Overview

This project implements a Multimodal Retrieval-Augmented Generation (RAG) system for answering questions from a company PDF report.

The system processes text, tables, images, charts, and diagrams.

Visual content is summarized using Qwen2-VL-2B. Sentence Transformers are used to create embeddings, and FAISS is used for local vector similarity search.

## Architecture

PDF Report
    |
    +--> Text extraction
    +--> Table extraction
    +--> Image extraction
              |
              v
        Qwen2-VL-2B
        Visual summaries
              |
              v
       LangChain Documents
              |
              v
    Sentence Transformers
              |
              v
            FAISS
              |
              v
        Similarity Retrieval
              |
              v
        Retrieved Evidence
              |
              v
        Qwen2-VL-2B
              |
              v
           Answer

## Technolog

In [35]:
from pathlib import Path
import shutil

PROJECT_DIR = Path("/content/multimodal-rag")

PROJECT_DIR.mkdir(exist_ok=True)

# Copy README
shutil.copy(
    "/content/README.md",
    PROJECT_DIR / "README.md"
)

# Copy requirements
shutil.copy(
    "/content/requirements.txt",
    PROJECT_DIR / "requirements.txt"
)

# Copy FAISS index
FAISS_DEST = PROJECT_DIR / "novacore_faiss_index"

if FAISS_DEST.exists():
    shutil.rmtree(FAISS_DEST)

shutil.copytree(
    "/content/novacore_faiss_index",
    FAISS_DEST
)

print("✅ Project folder created")
print(PROJECT_DIR)

✅ Project folder created
/content/multimodal-rag
